In [27]:
# LOAD DATASETS
import pandas as pd
crop_df = pd.read_csv("Dataset_X.csv")
soil_df = pd.read_csv("Dataset_Y.csv")

print(crop_df.shape, soil_df.shape)

(236496, 9) (695, 10)


In [28]:
# Normalize text (safety)
for col in ["state", "district", "crop"]:
    crop_df[col] = crop_df[col].str.strip().str.lower()
    soil_df[col] = soil_df[col].str.strip().str.lower()

In [29]:
crop_df

,state,district,year,season,crop,area,production,yield_log,risk
0,andaman and nicobar islands,nicobars,2000,kharif,arecanut,1254.0,2000.0000,0.953547,1
1,andaman and nicobar islands,nicobars,2000,kharif,rice,102.0,321.0000,1.422399,0
2,andaman and nicobar islands,nicobars,2000,annual,banana,176.0,641.0000,1.535155,0
3,andaman and nicobar islands,nicobars,2000,annual,cashew,720.0,165.0000,0.206336,2
4,andaman and nicobar islands,nicobars,2000,annual,coconut,18168.0,477860.7076,3.306972,0
...,...,...,...,...,...,...,...,...,...
236491,west bengal,purulia,2014,summer,rice,306.0,801.0000,1.285824,0
236492,west bengal,purulia,2014,summer,sesamum,627.0,463.0000,0.552986,1
236493,west bengal,purulia,2014,annual,sugarcane,324.0,16250.0000,3.934847,0
236494,west bengal,purulia,2014,rabi,rice,138371.0,477860.7076,1.493685,0


In [30]:
soil_df

,state,district,crop,latitude,longitude,n,p,k,ph,soil_fertility
0,maharashtra,"ahilya nagar, maharashtra, india",sugarcane,19.094829,74.747979,1.0361,2.6913,2.2111,7.0752,1.979500
1,maharashtra,"akola, maharashtra, india",cotton,20.700216,77.008168,1.0038,2.6580,2.8295,7.0876,2.163767
2,maharashtra,"amravati, maharashtra, india",cotton,20.931982,77.752304,1.0107,1.4771,2.6801,6.7714,1.722633
3,maharashtra,"chhatrapati sambhaji nagar, maharashtra, india",cotton,19.875754,75.339320,1.0198,2.5394,2.7132,7.1904,2.090800
4,maharashtra,"beed, maharashtra 431122, india",cotton,18.990088,75.753132,1.4096,2.0280,2.4846,7.1908,1.974067
...,...,...,...,...,...,...,...,...,...,...
690,jammu and kashmir,udhampur 182101,maize,32.915985,75.141617,2.2599,2.0020,1.8207,7.0000,2.027533
691,ladakh,kargil 194103,barley,34.553852,76.134894,0.0000,1.7143,1.8572,7.2858,1.190500
692,ladakh,leh,barley,34.152586,77.577054,0.0000,2.0609,2.4656,7.0916,1.508833
693,puducherry,"karaikal, puducherry, india",paddy,10.925440,79.838006,1.0000,2.4920,2.0880,6.9560,1.860000


In [31]:
# Check matching keys
common_districts = set(crop_df["state"]) & set(soil_df["state"])
common_crops = set(crop_df["crop"]) & set(soil_df["crop"])

print("Common states:", len(common_districts))
print("Common crops:", len(common_crops))

Common states: 31
Common crops: 29


In [37]:
soil_state = soil_df.groupby("state").agg({
    "n": "mean",
    "p": "mean",
    "k": "mean",
    "ph": "mean"
}).reset_index()

soil_state["soil_fertility"] = (
    soil_state["n"] + soil_state["p"] + soil_state["k"]
) / 3

soil_state

,state,n,p,k,ph,soil_fertility
0,andaman and nicobar islands,1.138133,1.087967,1.176000,6.805733,1.134033
1,andhra pradesh,1.285024,2.329892,2.429084,7.116136,2.014667
2,arunachal pradesh,2.049227,2.168319,1.976504,5.970973,2.064683
3,assam,2.411303,2.491528,1.663491,6.236522,2.188774
4,bihar,1.410669,2.756422,2.166022,6.928597,2.111038
5,chhattisgarh,1.261057,1.869829,2.424039,6.795114,1.851642
6,goa,2.017500,1.679500,1.959850,5.671950,1.885617
7,gujarat,1.215297,2.562815,2.542588,7.025764,2.106900
8,haryana,0.000000,2.481260,2.045090,7.049800,1.508783
9,himachal pradesh,1.408350,2.236892,2.301575,6.986542,1.982272


In [39]:
merged_df = pd.merge(
    crop_df,
    soil_state,
    on="state",
    how="left",
    validate="many_to_one",
    indicator=True
)
merged_df=merged_df.dropna()
merged_df = merged_df.drop(columns=["_merge"])
print(merged_df.columns)
merged_df

Index(['state', 'district', 'year', 'season', 'crop', 'area', 'production',
       'yield_log', 'risk', 'n', 'p', 'k', 'ph', 'soil_fertility'],
      dtype='str')


,state,district,year,season,crop,area,production,yield_log,risk,n,p,k,ph,soil_fertility
0,andaman and nicobar islands,nicobars,2000,kharif,arecanut,1254.0,2000.0000,0.953547,1,1.138133,1.087967,1.17600,6.805733,1.134033
1,andaman and nicobar islands,nicobars,2000,kharif,rice,102.0,321.0000,1.422399,0,1.138133,1.087967,1.17600,6.805733,1.134033
2,andaman and nicobar islands,nicobars,2000,annual,banana,176.0,641.0000,1.535155,0,1.138133,1.087967,1.17600,6.805733,1.134033
3,andaman and nicobar islands,nicobars,2000,annual,cashew,720.0,165.0000,0.206336,2,1.138133,1.087967,1.17600,6.805733,1.134033
4,andaman and nicobar islands,nicobars,2000,annual,coconut,18168.0,477860.7076,3.306972,0,1.138133,1.087967,1.17600,6.805733,1.134033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236491,west bengal,purulia,2014,summer,rice,306.0,801.0000,1.285824,0,1.602400,2.668890,2.10876,6.424235,2.126683
236492,west bengal,purulia,2014,summer,sesamum,627.0,463.0000,0.552986,1,1.602400,2.668890,2.10876,6.424235,2.126683
236493,west bengal,purulia,2014,annual,sugarcane,324.0,16250.0000,3.934847,0,1.602400,2.668890,2.10876,6.424235,2.126683
236494,west bengal,purulia,2014,rabi,rice,138371.0,477860.7076,1.493685,0,1.602400,2.668890,2.10876,6.424235,2.126683


In [41]:
df = merged_df[[
    "state",
    "district",
    "year",
    "season",
    "crop",
    "area",
    "production",
    "yield_log",
    "n",
    "p",
    "k",
    "ph",
    "soil_fertility",
    "risk"
]]
df

,state,district,year,season,crop,area,production,yield_log,n,p,k,ph,soil_fertility,risk
0,andaman and nicobar islands,nicobars,2000,kharif,arecanut,1254.0,2000.0000,0.953547,1.138133,1.087967,1.17600,6.805733,1.134033,1
1,andaman and nicobar islands,nicobars,2000,kharif,rice,102.0,321.0000,1.422399,1.138133,1.087967,1.17600,6.805733,1.134033,0
2,andaman and nicobar islands,nicobars,2000,annual,banana,176.0,641.0000,1.535155,1.138133,1.087967,1.17600,6.805733,1.134033,0
3,andaman and nicobar islands,nicobars,2000,annual,cashew,720.0,165.0000,0.206336,1.138133,1.087967,1.17600,6.805733,1.134033,2
4,andaman and nicobar islands,nicobars,2000,annual,coconut,18168.0,477860.7076,3.306972,1.138133,1.087967,1.17600,6.805733,1.134033,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236491,west bengal,purulia,2014,summer,rice,306.0,801.0000,1.285824,1.602400,2.668890,2.10876,6.424235,2.126683,0
236492,west bengal,purulia,2014,summer,sesamum,627.0,463.0000,0.552986,1.602400,2.668890,2.10876,6.424235,2.126683,1
236493,west bengal,purulia,2014,annual,sugarcane,324.0,16250.0000,3.934847,1.602400,2.668890,2.10876,6.424235,2.126683,0
236494,west bengal,purulia,2014,rabi,rice,138371.0,477860.7076,1.493685,1.602400,2.668890,2.10876,6.424235,2.126683,0


In [43]:
df.to_csv("Dataset_Z.csv", index=False)

print("Unified dataset ready")

Unified dataset ready
